# Exotic Options — Monte Carlo Deep Dive

A guided tour of the `exotic-options-mc` engine: simulating paths, pricing
path-dependent exotics, validating against closed forms, and measuring the
payoff of variance reduction.

Run from the repository root so the packages import cleanly.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt

from simulation.gbm import GBMSimulator
from pricing.asian import AsianOptionPricer
from pricing.barrier import BarrierOptionPricer
from pricing.lookback import LookbackOptionPricer
from variance_reduction.combined import CombinedVR, Contract
from benchmarks.analytical import bsm_price

S0, K, T, r, q, sigma = 100.0, 100.0, 1.0, 0.05, 0.0, 0.20

## 1. Simulate GBM paths

The exact log-Euler scheme is fully vectorized: 100k paths over 252 steps in
well under a second.

In [ ]:
g = GBMSimulator(seed=42)
paths = g.generate_paths(S0, T, r, q, sigma, n_paths=100_000, n_steps=252)
print('shape', paths.shape)
print('E[S_T] MC', paths[:, -1].mean(), 'theory', S0 * np.exp((r - q) * T))

plt.figure(figsize=(9, 4))
for path in paths[:40]:
    plt.plot(path, lw=0.6, alpha=0.5)
plt.title('Sample GBM paths'); plt.xlabel('step'); plt.ylabel('price'); plt.show()

## 2. Price an arithmetic Asian and validate the geometric leg

The arithmetic Asian has no closed form; the geometric Asian does
(Kemna-Vorst) and serves as both a benchmark and a control variate.

In [ ]:
ap = AsianOptionPricer('continuous')
arith = ap.price_arithmetic(paths, K, r, T, 'call')
geo = ap.price_geometric(paths, K, r, T, 'call')
geo_an = ap.price_geometric_analytical(S0, K, T, r, q, sigma, 'call')
print(f'Arithmetic Asian : {arith.price:.4f} +/- {arith.stderr:.4f}')
print(f'Geometric  Asian : {geo.price:.4f} (analytic {geo_an:.4f})')
print('Jensen: arithmetic >= geometric ->', arith.price > geo.price)

## 3. The payoff of control variates

Using the geometric Asian as a control collapses the standard error by orders
of magnitude for essentially no extra compute.

In [ ]:
cv = ap.price_with_control_variate(paths, K, r, T, q, sigma, S0, 'call')
print(f'Crude          : {arith.price:.4f} +/- {arith.stderr:.4f}')
print(f'Control variate: {cv.price:.4f} +/- {cv.stderr:.4f}')
print(f'Variance reduction ratio: {cv.extra["vr_ratio"]:.0f}x')

## 4. Full variance-reduction efficiency table

In [ ]:
table = CombinedVR(seed=7).efficiency_table(Contract(S0=S0, K=K, T=T, r=r, q=q, sigma=sigma), n_paths=40_000)
table

## 5. Barrier and lookback validation

Discrete monitoring is corrected (Brownian bridge for barriers, BGK shift for
lookbacks) so the Monte Carlo prices match the continuous closed forms.

In [ ]:
bp = BarrierOptionPricer()
uo_mc = bp.price_mc(paths, K, 120.0, r, T, 'up-and-out', 'call', sigma=sigma, brownian_bridge=True)
uo_an = bp.price_analytical(S0, K, 120.0, T, r, q, sigma, 'up-and-out', 'call')
print(f'Up-and-out call  MC {uo_mc.price:.4f}  RR {uo_an:.4f}')

lb = LookbackOptionPricer()
lc_mc = lb.price_floating_strike(paths, r, T, 'call', sigma=sigma, continuity_correction=True)
lc_an = lb.floating_call_analytical(S0, T, r, q, sigma)
print(f'Floating lookback MC {lc_mc.price:.4f}  GSG {lc_an:.4f}')

## 6. Convergence: the O(1/sqrt(N)) law

In [ ]:
from diagnostics.convergence import ConvergenceAnalyzer
ca = ConvergenceAnalyzer()
def pricer(n):
    p = GBMSimulator(seed=1).generate_paths(S0, T, r, q, sigma, int(n), 100)
    return ap.price_arithmetic(p, K, r, T, 'call')
df = ca.price_vs_paths(pricer, [1_000, 5_000, 10_000, 50_000, 100_000])
print('fitted slope (expect ~ -0.5):', round(ca.convergence_rate(df), 3))

plt.figure(figsize=(8, 4))
plt.loglog(df['n_paths'], df['stderr'], 'o-')
plt.xlabel('n_paths'); plt.ylabel('standard error'); plt.title('Monte Carlo convergence'); plt.grid(True, which='both', alpha=0.3); plt.show()